# Event Study cu Market Model

Citeste acest notebook dupa `event_study_explicat_ro.ipynb`.

Primul notebook explica exact modelul din codul actual: o regresie liniara doar cu intercept.
Acest al doilea notebook explica varianta mai standard din finante, numita **market model**.

Ideea centrala este aceasta:

- in loc sa spunem doar ca randamentul normal este media istorica
- spunem ca randamentul unui activ depinde partial de miscarea pietei


## 1. De ce avem nevoie de un model mai bogat?

Modelul simplu din primul notebook este:

$$
r_t = \alpha + \varepsilon_t
$$

Acolo, randamentul normal este aceeasi valoare in fiecare zi: media istorica.

Dar in realitate, daca piata intreaga urca sau coboara puternic, este normal ca si activul nostru sa fie influentat.
De aceea, in literatura de event study apare foarte des modelul:

$$
R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}
$$

Unde:

- $R_{i,t}$ = randamentul activului $i$ in ziua $t$
- $R_{m,t}$ = randamentul pietei in ziua $t$
- $\alpha_i$ = componenta medie proprie activului
- $\beta_i$ = sensibilitatea activului la piata
- $\varepsilon_{i,t}$ = partea care nu este explicata de piata


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## 2. Un mic set de date didactic

Mai jos folosim un exemplu mic, construit special pentru invatare.

Avem doua serii de randamente:

- randamentul activului
- randamentul pietei

Primele observatii sunt pentru **fereastra de estimare**.
Ultimele observatii vor fi folosite drept **fereastra de eveniment**.


In [ ]:
dates = pd.date_range("2024-01-02", periods=12, freq="B")

market_returns = pd.Series(
    [0.004, -0.002, 0.006, 0.001, -0.003, 0.005, 0.002, -0.004, 0.003, 0.001, 0.008, -0.007],
    index=dates,
    name="Randament piata",
)

asset_returns = pd.Series(
    [0.006, -0.004, 0.009, 0.002, -0.005, 0.007, 0.004, -0.006, 0.005, 0.003, 0.018, -0.014],
    index=dates,
    name="Randament activ",
)

df = pd.concat([asset_returns, market_returns], axis=1)
df


## 3. Fereastra de estimare si fereastra de eveniment

Vom folosi:

- primele 8 observatii pentru estimare
- ultimele 4 observatii pentru eveniment


In [ ]:
estimation_df = df.iloc[:8].copy()
event_df = df.iloc[8:].copy()

print("Fereastra de estimare")
display(estimation_df)

print("Fereastra de eveniment")
display(event_df)


## 4. Cum se antreneaza regresia liniara

Modelul este:

$$
R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}
$$

Acum vrem sa estimam $\alpha_i$ si $\beta_i$ din datele istorice.

Formulele OLS sunt:

$$
\hat{\beta}_i = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}
$$

$$
\hat{\alpha}_i = \overline{R_i} - \hat{\beta}_i \overline{R_m}
$$

Interpretare:

- $\beta$ spune cat de sensibil este activul la piata
- daca $\beta = 1$, activul se misca aproximativ la fel ca piata
- daca $\beta > 1$, activul reactioneaza mai puternic
- daca $\beta < 1$, activul reactioneaza mai slab


In [ ]:
R_i = estimation_df["Randament activ"]
R_m = estimation_df["Randament piata"]

beta_hat = np.cov(R_i, R_m, ddof=1)[0, 1] / np.var(R_m, ddof=1)
alpha_hat = R_i.mean() - beta_hat * R_m.mean()

pd.Series(
    {
        "alpha estimat": alpha_hat,
        "alpha estimat (%)": alpha_hat * 100,
        "beta estimat": beta_hat,
    }
)


### Exemplu foarte mic, facut de mana

Daca piata are randamentele `[1%, 2%, 3%]` si activul are `[2%, 3%, 4%]`, intuitia este ca activul se comporta cam ca piata, dar putin deasupra ei.

In acest caz, o regresie liniara ar produce aproximativ:

- $\beta \approx 1$
- $\alpha \approx 1\%$

Adica modelul ar spune: activul urmeaza piata aproape unu-la-unu, dar are si un mic plus propriu.


In [ ]:
market_toy = np.array([0.01, 0.02, 0.03])
asset_toy = np.array([0.02, 0.03, 0.04])

beta_toy = np.cov(asset_toy, market_toy, ddof=1)[0, 1] / np.var(market_toy, ddof=1)
alpha_toy = asset_toy.mean() - beta_toy * market_toy.mean()

print(f"beta toy = {beta_toy:.3f}")
print(f"alpha toy = {alpha_toy:.3f}")


## 5. Cum se face predictia

Dupa antrenare, pentru fiecare zi din fereastra de eveniment, prezicem randamentul normal astfel:

$$
\widehat{R}_{i,t} = \hat{\alpha}_i + \hat{\beta}_i R_{m,t}
$$

Observa diferenta fata de primul notebook:

- in modelul simplu, predictia era aceeasi in fiecare zi
- in market model, predictia se schimba de la o zi la alta pentru ca depinde de ce a facut piata


In [ ]:
event_df = event_df.copy()
event_df["Randament normal prezis"] = alpha_hat + beta_hat * event_df["Randament piata"]
event_df["Randament anormal"] = event_df["Randament activ"] - event_df["Randament normal prezis"]

event_df


Formula pentru randamentul anormal ramane aceeasi idee ca in primul notebook:

$$
AR_t = R_{i,t} - \widehat{R}_{i,t}
$$

Doar ca acum randamentul normal prezis este mai inteligent, pentru ca tine cont de miscare pietei.


## 6. `t-test` si `p-value` in market model

Dupa ce avem randamentele anormale, putem repeta aceeasi idee statistica:

1. calculam media randamentelor anormale
2. verificam daca aceasta medie este suficient de departe de zero

Pentru notebook-ul didactic de mai jos folosim abaterea standard a reziduurilor din fereastra de estimare.

Reziduurile istorice sunt:

$$
\hat{\varepsilon}_{i,t} = R_{i,t} - (\hat{\alpha}_i + \hat{\beta}_i R_{m,t})
$$

Apoi folosim:

$$
t = \frac{\overline{AR}}{s_{\varepsilon} / \sqrt{n}}
$$

unde $s_{\varepsilon}$ este abaterea standard a reziduurilor din fereastra de estimare.


In [ ]:
estimation_residuals = estimation_df["Randament activ"] - (alpha_hat + beta_hat * estimation_df["Randament piata"])
residual_std = estimation_residuals.std(ddof=1)

abnormal_returns = event_df["Randament anormal"]
mean_ar = abnormal_returns.mean()
n = len(abnormal_returns)
t_stat = mean_ar / (residual_std / np.sqrt(n))
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))

pd.Series(
    {
        "Media randamentelor anormale": mean_ar,
        "Media randamentelor anormale (%)": mean_ar * 100,
        "Dev. std. reziduuri istorice": residual_std,
        "t-statistic": t_stat,
        "p-value": p_value,
        "Semnificativ la prag de 5%": p_value < 0.05,
    }
)


## 7. Diferenta dintre cele doua notebook-uri

Primul notebook:

- este fidel codului actual din proiect
- foloseste doar media randamentelor istorice
- este excelent pentru intuitia de baza

Al doilea notebook:

- este mai aproape de forma academica standard din finante
- introduce randamentul pietei
- estimeaza `alpha` si `beta`
- produce un randament normal care se schimba in functie de piata

Ordinea buna de invatare este exact asta:

1. intai modelul simplu
2. apoi market model
